In [181]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform


# **Data Preprocessing**

In [182]:
df = pd.read_csv('/content/heart_cleveland_upload.csv')
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,condition
0,69,1,0,160,234,1,2,131,0,0.1,1,1,0,0
1,69,0,0,140,239,0,0,151,0,1.8,0,2,0,0
2,66,0,0,150,226,0,0,114,0,2.6,2,0,0,0
3,65,1,0,138,282,1,2,174,0,1.4,1,1,0,1
4,64,1,0,110,211,0,2,144,1,1.8,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,40,1,3,152,223,0,0,181,0,0.0,0,0,2,1
293,39,1,3,118,219,0,0,140,0,1.2,1,0,2,1
294,35,1,3,120,198,0,0,130,1,1.6,1,0,2,1
295,35,0,3,138,183,0,0,182,0,1.4,0,0,0,0


In [183]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        297 non-null    int64  
 1   sex        297 non-null    int64  
 2   cp         297 non-null    int64  
 3   trestbps   297 non-null    int64  
 4   chol       297 non-null    int64  
 5   fbs        297 non-null    int64  
 6   restecg    297 non-null    int64  
 7   thalach    297 non-null    int64  
 8   exang      297 non-null    int64  
 9   oldpeak    297 non-null    float64
 10  slope      297 non-null    int64  
 11  ca         297 non-null    int64  
 12  thal       297 non-null    int64  
 13  condition  297 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 32.6 KB


In [184]:
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,condition
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000
mean,54.542088,0.676768,2.158249,131.693603,247.350168,0.144781,0.996633,149.599327,0.326599,1.055556,0.602694,0.676768,0.835017,0.461279
std,9.049736,0.468500,0.964859,17.762806,51.997583,0.352474,0.994914,22.941562,0.469761,1.166123,0.618187,0.938965,0.956690,0.499340
min,29.000000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.000000,0.000000,2.000000,120.000000,211.000000,0.000000,0.000000,133.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,56.000000,1.000000,2.000000,130.000000,243.000000,0.000000,1.000000,153.000000,0.000000,0.800000,1.000000,0.000000,0.000000,0.000000
75%,61.000000,1.000000,3.000000,140.000000,276.000000,0.000000,2.000000,166.000000,1.000000,1.600000,1.000000,1.000000,2.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,3.000000,2.000000,1.000000


In [185]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [186]:
df.shape

(297, 14)

In [187]:
df['condition'].value_counts()

,count
condition,
0,160
1,137


In [188]:
'''Check for duplicates
   Remove duplicates if any '''

df.duplicated().sum()
df = df.drop_duplicates()


In [189]:
X = df.drop('condition', axis=1)
y = df['condition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def feature_engineering(df):
    df = df.copy()

    ''' Age Group Feature '''
    df['age_group'] = pd.cut(df['age'],
                              bins=[0, 40, 55, 70, 100],
                              labels=['young', 'middle', 'senior', 'elderly'])

    ''' Blood Pressure Category '''
    df['bp_category'] = pd.cut(df['trestbps'],
                                 bins=[0, 120, 140, 200],
                                 labels=['normal', 'elevated', 'high'])

    ''' Cholesterol Category '''
    df['chol_category'] = pd.cut(df['chol'],
                                   bins=[0, 200, 240, 400],
                                   labels=['normal', 'borderline', 'high'])

    ''' Interaction Features '''
    df['age_chol_interaction'] = df['age'] * df['chol']
    df['age_thalach_interaction'] = df['age'] * df['thalach']

    ''' Polynomial Features (age squared) '''
    df['age_squared'] = df['age'] ** 2

    ''' Risk Score '''
    df['risk_score'] = (df['age'] * 0.3 +
                         df['chol'] * 0.2 +
                         df['trestbps'] * 0.25 +
                         df['oldpeak'] * 0.25)

    return df

X_train_fe = feature_engineering(X_train)
X_test_fe = feature_engineering(X_test)

''' Encoding the Categorical features '''
X_train_fe = pd.get_dummies(X_train_fe, columns=['age_group', 'bp_category', 'chol_category'], drop_first=True)
X_test_fe = pd.get_dummies(X_test_fe, columns=['age_group', 'bp_category', 'chol_category'], drop_first=True)

X_test_fe = X_test_fe.reindex(columns=X_train_fe.columns, fill_value=0)

print("✅ Features after engineering:", X_train_fe.shape[1])
print("Feature names:", X_train_fe.columns.tolist())

''' Handle class Imbalance using SMOTE '''
print("\nBefore SMOTE:")
print("Training set shape:", X_train_fe.shape)
print("Class distribution:", Counter(y_train))

smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_fe, y_train)

print("\nAfter SMOTE:")
print("Training set shape:", X_train_smote.shape)
print("Class distribution:", Counter(y_train_smote))

''' Scaling the feature '''
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test_fe)

print("\nScaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)



✅ Features after engineering: 24
Feature names: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'age_chol_interaction', 'age_thalach_interaction', 'age_squared', 'risk_score', 'age_group_middle', 'age_group_senior', 'age_group_elderly', 'bp_category_elevated', 'bp_category_high', 'chol_category_borderline', 'chol_category_high']

Before SMOTE:
Training set shape: (237, 24)
Class distribution: Counter({0: 128, 1: 109})

After SMOTE:
Training set shape: (256, 24)
Class distribution: Counter({1: 128, 0: 128})

Scaling complete.
X_train_scaled shape: (256, 24)
X_test_scaled shape: (60, 24)


# **Decision Tree**

In [190]:
dt_classifier = DecisionTreeClassifier(random_state=42)

dt_classifier.fit(X_train_scaled, y_train_smote)

y_pred_train_dt = dt_classifier.predict(X_train_scaled)
y_pred_test_dt = dt_classifier.predict(X_test_scaled)
y_pred_proba_dt = dt_classifier.predict_proba(X_test_scaled)[:, 1]

print("\nTraining Accuracy:", accuracy_score(y_train_smote, y_pred_train_dt))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_dt))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_dt))


Training Accuracy: 1.0
Testing Accuracy: 0.8

ROC-AUC Score: 0.7991071428571428

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        32
           1       0.79      0.79      0.79        28

    accuracy                           0.80        60
   macro avg       0.80      0.80      0.80        60
weighted avg       0.80      0.80      0.80        60


Confusion Matrix:
[[26  6]
 [ 6 22]]


# **Random Forest**

In [191]:

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_classifier.fit(X_train_scaled, y_train_smote)

y_pred_train_rf = rf_classifier.predict(X_train_scaled)
y_pred_test_rf = rf_classifier.predict(X_test_scaled)
y_pred_proba_rf = rf_classifier.predict_proba(X_test_scaled)[:, 1]

print("\nTraining Accuracy:", accuracy_score(y_train_smote, y_pred_train_rf))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_rf))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_rf))




Training Accuracy: 1.0
Testing Accuracy: 0.8333333333333334

ROC-AUC Score: 0.9319196428571428

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.94      0.86        32
           1       0.91      0.71      0.80        28

    accuracy                           0.83        60
   macro avg       0.85      0.83      0.83        60
weighted avg       0.85      0.83      0.83        60


Confusion Matrix:
[[30  2]
 [ 8 20]]


# **Gradient Boost**

In [192]:
gb_classifier = GradientBoostingClassifier(random_state=42)

gb_classifier.fit(X_train_scaled, y_train_smote)

y_pred_train_gb = gb_classifier.predict(X_train_scaled)
y_pred_test_gb = gb_classifier.predict(X_test_scaled)
y_pred_proba_gb = gb_classifier.predict_proba(X_test_scaled)[:, 1]


print("\nTraining Accuracy:", accuracy_score(y_train_smote, y_pred_train_gb))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_gb))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba_gb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_gb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_gb))




Training Accuracy: 1.0
Testing Accuracy: 0.8166666666666667

ROC-AUC Score: 0.921875

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.91      0.84        32
           1       0.87      0.71      0.78        28

    accuracy                           0.82        60
   macro avg       0.83      0.81      0.81        60
weighted avg       0.82      0.82      0.81        60


Confusion Matrix:
[[29  3]
 [ 8 20]]


# **XG boost**

In [193]:
xgb_classifier = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

xgb_classifier.fit(X_train_smote, y_train_smote)

y_pred_train_xgb = xgb_classifier.predict(X_train_smote)
y_pred_test_xgb = xgb_classifier.predict(X_test_scaled)
y_pred_proba_xgb = xgb_classifier.predict_proba(X_test_scaled)[:, 1]


print("\nTraining Accuracy:", accuracy_score(y_train_smote, y_pred_train_xgb))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_xgb))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_xgb))




Training Accuracy: 1.0
Testing Accuracy: 0.6666666666666666

ROC-AUC Score: 0.86328125

Classification Report:
              precision    recall  f1-score   support

           0       0.62      1.00      0.76        32
           1       1.00      0.29      0.44        28

    accuracy                           0.67        60
   macro avg       0.81      0.64      0.60        60
weighted avg       0.79      0.67      0.61        60


Confusion Matrix:
[[32  0]
 [20  8]]


# Tuning Gradient Boosting using RandomSearchCV

In [200]:



''' Defining parameter distributions for Gradient Boosting '''
param_dist_gb = {
    'n_estimators': randint(50, 200),
    'learning_rate': uniform(0.05, 0.15),
    'max_depth': randint(3, 6),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'subsample': uniform(0.7, 0.3),
}

gb_classifier_random = GradientBoostingClassifier(random_state=42)

random_search_gb = RandomizedSearchCV(
    estimator=gb_classifier_random,
    param_distributions=param_dist_gb,
    n_iter=30,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=False
)


random_search_gb.fit(X_train_scaled, y_train_smote)


print(f"\nBest Cross-Validation ROC-AUC Score: {random_search_gb.best_score_:.4f}")

best_gb_model_random = random_search_gb.best_estimator_

y_pred_test_gb_random = best_gb_model_random.predict(X_test_scaled)
y_pred_proba_gb_random = best_gb_model_random.predict_proba(X_test_scaled)[:, 1]

print("\nTEST SET EVALUATION")
print(f"\nTesting Accuracy: {accuracy_score(y_test, y_pred_test_gb_random):.4f}")
print(f"Testing ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_gb_random):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_gb_random))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_gb_random))

Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best Cross-Validation ROC-AUC Score: 0.8534

TEST SET EVALUATION

Testing Accuracy: 0.7833
Testing ROC-AUC Score: 0.9185

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.88      0.81        32
           1       0.83      0.68      0.75        28

    accuracy                           0.78        60
   macro avg       0.79      0.78      0.78        60
weighted avg       0.79      0.78      0.78        60


Confusion Matrix:
[[28  4]
 [ 9 19]]


# Tuning XG Boost using RandomSearchCV

In [201]:

''' Defining parameter distributions for XGBoost '''
param_dist_xgb = {
    'n_estimators': randint(100, 500),
    'learning_rate': uniform(0.01, 0.29),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 6),
    'gamma': uniform(0, 0.5),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1)
}

xgb_classifier_tuned = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_classifier_tuned,
    param_distributions=param_dist_xgb,
    n_iter=50,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    random_state=42
)


random_search_xgb.fit(X_train_smote, y_train_smote)

print("XGBOOST - RANDOMIZEDSEARCHCV RESULTS")
print("Best Cross-Validation ROC-AUC Score:", random_search_xgb.best_score_)

'''Evaluating the best model'''
best_xgb_model = random_search_xgb.best_estimator_
y_pred_test_xgb_tuned = best_xgb_model.predict(X_test_scaled)
y_pred_proba_xgb_tuned = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

print("\nTEST SET EVALUATION")
print("\nTesting Accuracy:", accuracy_score(y_test, y_pred_test_xgb_tuned))
print("Testing ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba_xgb_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test_xgb_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test_xgb_tuned))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
XGBOOST - RANDOMIZEDSEARCHCV RESULTS
Best Cross-Validation ROC-AUC Score: 0.8846272189349111

TEST SET EVALUATION

Testing Accuracy: 0.6333333333333333
Testing ROC-AUC Score: 0.8454241071428571

Classification Report:
              precision    recall  f1-score   support

           0       0.59      1.00      0.74        32
           1       1.00      0.21      0.35        28

    accuracy                           0.63        60
   macro avg       0.80      0.61      0.55        60
weighted avg       0.78      0.63      0.56        60


Confusion Matrix:
[[32  0]
 [22  6]]


# Comparing all the Model

In [196]:
models_comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Gradient Boosting',
              'XGBoost', 'XGBoost (Tuned)','GB_Tuned'],
    'Test_Accuracy': [
        accuracy_score(y_test, y_pred_test_dt),
        accuracy_score(y_test, y_pred_test_rf),
        accuracy_score(y_test, y_pred_test_gb),
        accuracy_score(y_test, y_pred_test_xgb),
        accuracy_score(y_test, y_pred_test_xgb_tuned),
        accuracy_score(y_test, y_pred_test_gb_random)
    ],
    'ROC_AUC': [
        roc_auc_score(y_test, y_pred_proba_dt),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_gb),
        roc_auc_score(y_test, y_pred_proba_xgb),
        roc_auc_score(y_test, y_pred_proba_xgb_tuned),
        roc_auc_score(y_test, y_pred_proba_gb_random)
    ]
})

print("MODEL COMPARISON SUMMARY")
print(models_comparison.to_string(index=False))
print("\nBest Model:", models_comparison.loc[models_comparison['ROC_AUC'].idxmax(), 'Model'])

MODEL COMPARISON SUMMARY
            Model  Test_Accuracy  ROC_AUC
    Decision Tree       0.800000 0.799107
    Random Forest       0.833333 0.931920
Gradient Boosting       0.816667 0.921875
          XGBoost       0.666667 0.863281
  XGBoost (Tuned)       0.633333 0.845424
         GB_Tuned       0.783333 0.918527

Best Model: Random Forest


Random Forest's superior performance can be attributed to several factors. The bagging approach builds multiple independent trees in parallel, each trained on different bootstrap samples with random feature subsets, which creates high diversity and reduces overfitting. For your heart disease dataset, this diversity likely captured various patterns without overfitting to noise. Random Forest is particularly robust on small-to-medium datasets where sequential boosting methods might overfit.​

Strengths:

* Less prone to overfitting due to independent tree construction​

* Robust to outliers and noise in the data​

* Works well out-of-the-box with minimal tuning​

* Excellent for small-to-medium datasets



1.   Random Forest trains fastest among ensembles because all trees are built independently in parallel. Gradient Boosting is slower as each tree depends on the previous one's errors. XGBoost optimizes this by parallelizing within each tree but still remains sequential overall.

2.   Random Forest has the lowest overfitting tendency due to its bagging approach and random feature selection at each split. Each tree overfits in different ways, which cancels out when aggregated. Gradient Boosting and XGBoost are more prone to overfitting because they sequentially focus on hard-to-predict instances.The results confirm this, default Random Forest generalized better than tuned boosting models.

3. Decision Trees are most interpretable as you can visualize the entire decision path. Random Forest provides good feature importance rankings while sacrificing individual tree interpretability. Gradient Boosting and XGBoost are harder to interpret due to sequential error correction and complex interactions between trees.​



